# Price Bucket Vector (PBV) Strategy

## Concept

At each LOB snapshot we maintain **5 running vectors** — one per price bucket relative to the midprice — that accumulate weighted order book activity within a minute window.

```
v_mid        ← order imbalance at best bid/ask (around the midpoint)
v_ask_near   ← pressure at best ask (AskPrice_1), closest level above mid
v_ask_far    ← pressure at outer ask levels (AskPrice 2-5), further above mid
v_bid_near   ← pressure at best bid (BidPrice_1), closest level below mid
v_bid_far    ← pressure at outer bid levels (BidPrice 2-5), further below mid
```

Their **cumulative sums within each minute**, plus first and second derivatives, form the trading signal.

**BUY signal logic:** execute when ask-side pressure is decelerating (wall weakening)  
**SELL signal logic:** execute when bid-side pressure is decelerating (support weakening)  
**Fallback:** execute at the last tick of the minute if no signal fires

In [ ]:
# ============================================================
# CELL 1: Imports & Data Load
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# NOTE: scale to other stocks by changing this path
STOCK = "AMZN"
train_raw = pd.read_csv(f"training_datasets/{STOCK}_5levels_train.csv")

# Parse time; floor to minute for grouping
train_raw["Time_dt"] = pd.to_datetime(train_raw["Time"], format="%H:%M:%S.%f")
train_raw["Minute"]  = train_raw["Time_dt"].dt.floor("min")

print(f"Shape: {train_raw.shape}")
print(f"Columns: {list(train_raw.columns)}")
print(f"Minutes in dataset: {train_raw['Minute'].nunique()}")
train_raw.head(5)

In [ ]:
# ============================================================
# CELL 2: Instantaneous Bucket Value Computation
# ============================================================
# Each bucket captures a different type of pressure relative to mid.
# Pressure = size / distance_from_mid  (larger size closer to mid = more pressure)
#
# v_mid      : signed imbalance at the top of book (bid - ask) / (bid + ask)
#              positive  → more bid volume → upward pressure
# v_ask_near : inverse-distance-weighted size at best ask (level 1 above mid)
# v_ask_far  : sum of inverse-distance-weighted sizes at ask levels 2-5
# v_bid_near : inverse-distance-weighted size at best bid (level 1 below mid)
# v_bid_far  : sum of inverse-distance-weighted sizes at bid levels 2-5

EPS = 1e-6  # avoid division by zero

def compute_bucket_row(row):
    mid = row["MidPrice"]

    # --- mid: signed top-of-book imbalance ---
    b1, a1 = row["BidSize_1"], row["AskSize_1"]
    v_mid = (b1 - a1) / (b1 + a1 + EPS)

    # --- ask near: level 1 above mid ---
    dist_ask1 = max(row["AskPrice_1"] - mid, EPS)
    v_ask_near = row["AskSize_1"] / dist_ask1

    # --- ask far: levels 2-5 above mid ---
    v_ask_far = sum(
        row[f"AskSize_{i}"] / max(row[f"AskPrice_{i}"] - mid, EPS)
        for i in range(2, 6)
    )

    # --- bid near: level 1 below mid ---
    dist_bid1 = max(mid - row["BidPrice_1"], EPS)
    v_bid_near = row["BidSize_1"] / dist_bid1

    # --- bid far: levels 2-5 below mid ---
    v_bid_far = sum(
        row[f"BidSize_{i}"] / max(mid - row[f"BidPrice_{i}"], EPS)
        for i in range(2, 6)
    )

    return v_mid, v_ask_near, v_ask_far, v_bid_near, v_bid_far


BUCKET_COLS = ["v_mid", "v_ask_near", "v_ask_far", "v_bid_near", "v_bid_far"]

bucket_vals = train_raw.apply(
    lambda r: compute_bucket_row(r), axis=1, result_type="expand"
)
bucket_vals.columns = BUCKET_COLS

train = pd.concat([train_raw, bucket_vals], axis=1)
print("Bucket columns added.")
train[BUCKET_COLS].describe()

In [ ]:
# ============================================================
# CELL 3: Cumulative Vectors + Derivatives per Minute
# ============================================================
# Within each minute the vectors accumulate (reset to 0 at each new minute).
# We then compute:
#   _cum   : running sum within the minute (the 'vector state')
#   _d1    : first derivative  of _cum  (velocity  — how fast the state is changing)
#   _d2    : second derivative of _cum  (acceleration — is the velocity itself speeding up?)

for col in BUCKET_COLS:
    # Cumulative sum resets every minute via groupby
    train[f"{col}_cum"] = train.groupby("Minute")[col].cumsum()

    # Derivatives — NaN at minute boundaries is expected and iclaud/ntentional
    train[f"{col}_d1"] = train.groupby("Minute")[f"{col}_cum"].diff()
    train[f"{col}_d2"] = train.groupby("Minute")[f"{col}_d1"].diff()

CUM_COLS = [f"{c}_cum" for c in BUCKET_COLS]
D1_COLS  = [f"{c}_d1"  for c in BUCKET_COLS]
D2_COLS  = [f"{c}_d2"  for c in BUCKET_COLS]

print("Cumulative vectors and derivatives computed.")
print(f"New columns: {CUM_COLS + D1_COLS + D2_COLS}")
train[CUM_COLS + D2_COLS].describe()

In [ ]:
# ============================================================
# CELL 4: Visualise Bucket Vectors for a Sample Minute
# ============================================================
# Pick an arbitrary minute to inspect the raw vectors, cumulative state,
# and their second derivatives side by side.

SAMPLE_MINUTE = train["Minute"].unique()[5]  # change index to explore others
sample = train[train["Minute"] == SAMPLE_MINUTE].reset_index(drop=True)

fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.3)

ax_raw  = fig.add_subplot(gs[0, :])
ax_cum  = fig.add_subplot(gs[1, :])
ax_d1   = fig.add_subplot(gs[2, 0])
ax_d2   = fig.add_subplot(gs[2, 1])

colors = ["black", "red", "salmon", "steelblue", "lightblue"]

for col, c in zip(BUCKET_COLS, colors):
    ax_raw.plot(sample[col].values, label=col, color=c, alpha=0.8)
    ax_cum.plot(sample[f"{col}_cum"].values, label=col, color=c, alpha=0.8)
    ax_d1.plot(sample[f"{col}_d1"].values, label=col, color=c, alpha=0.8)
    ax_d2.plot(sample[f"{col}_d2"].values, label=col, color=c, alpha=0.8)

ax_raw.set_title(f"Instantaneous bucket values — {SAMPLE_MINUTE}")
ax_cum.set_title("Cumulative bucket vectors (within-minute state)")
ax_d1.set_title("1st derivative (velocity)")
ax_d2.set_title("2nd derivative (acceleration)")

for ax in [ax_raw, ax_cum, ax_d1, ax_d2]:
    ax.legend(fontsize=7, ncol=5)
    ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
    ax.set_xlabel("Tick within minute")

plt.suptitle("Price Bucket Vectors — Single Minute Inspection", fontsize=13, y=1.01)
plt.show()

In [ ]:
# ============================================================
# CELL 5: Signal Definition
# ============================================================
# BUY signal: execute when the ask wall is weakening AND bid imbalance tilts up.
#   - v_ask_near_d2 < 0  → near ask pressure is decelerating (wall softening)
#   - v_mid_cum    > 0   → cumulative imbalance favours the bid side
#
# SELL signal: mirror logic.
#   - v_bid_near_d2 < 0  → near bid pressure is decelerating (support weakening)
#   - v_mid_cum    < 0   → cumulative imbalance favours the ask side
#
# FALLBACK: if no signal has fired by FALLBACK_SECONDS, execute at the next tick.

FALLBACK_SECONDS = 50   # execute in last 10s of each minute if no signal
SIDE = "BUY"            # toggle to "SELL" for the sell algorithm

def within_minute_seconds(row):
    """Seconds elapsed since the start of this minute."""
    return (row["Time_dt"] - row["Minute"]).total_seconds()


def buy_signal(row):
    elapsed = within_minute_seconds(row)
    if elapsed >= FALLBACK_SECONDS:
        return True
    # Skip the first 2 ticks — derivatives not meaningful yet (NaN)
    if pd.isna(row["v_ask_near_d2"]) or pd.isna(row["v_mid_cum"]):
        return False
    ask_weakening     = row["v_ask_near_d2"] < 0
    imbalance_up      = row["v_mid_cum"]      > 0
    return bool(ask_weakening and imbalance_up)


def sell_signal(row):
    elapsed = within_minute_seconds(row)
    if elapsed >= FALLBACK_SECONDS:
        return True
    if pd.isna(row["v_bid_near_d2"]) or pd.isna(row["v_mid_cum"]):
        return False
    bid_weakening     = row["v_bid_near_d2"] < 0
    imbalance_down    = row["v_mid_cum"]      < 0
    return bool(bid_weakening and imbalance_down)


SIGNAL_FN      = buy_signal  if SIDE == "BUY"  else sell_signal
EXEC_PRICE_COL = "AskPrice_1" if SIDE == "BUY" else "BidPrice_1"

print(f"Side: {SIDE} | Execution price column: {EXEC_PRICE_COL}")
print(f"Fallback fires after {FALLBACK_SECONDS}s within each minute")

In [ ]:
# ============================================================
# CELL 6: Simulate PBV Execution vs TWAP Benchmark
# ============================================================
# TWAP benchmark: first tick of each minute (same as standard TWAP).
# PBV strategy:  first tick where the signal fires.

results = []

for minute, grp in train.groupby("Minute"):
    grp = grp.reset_index(drop=True)

    # TWAP reference: first tick of the minute
    twap_price = grp.iloc[0][EXEC_PRICE_COL]

    # PBV: first row where signal fires
    grp["signal"] = grp.apply(SIGNAL_FN, axis=1)
    fired_rows = grp[grp["signal"]]

    if fired_rows.empty:
        # No signal and fallback didn't trigger — execute at last tick
        fired_row = grp.iloc[-1]
        fallback  = True
    else:
        fired_row = fired_rows.iloc[0]
        fallback  = within_minute_seconds(fired_row) >= FALLBACK_SECONDS

    pbv_price   = fired_row[EXEC_PRICE_COL]
    exec_second = within_minute_seconds(fired_row)

    # For BUY: improvement = lower price is better  → twap - pbv
    # For SELL: improvement = higher price is better → pbv - twap
    improvement = (twap_price - pbv_price) if SIDE == "BUY" else (pbv_price - twap_price)

    results.append({
        "Minute"      : minute,
        "TWAP_price"  : twap_price,
        "PBV_price"   : pbv_price,
        "improvement" : improvement,
        "exec_second" : exec_second,
        "fallback"    : fallback,
        "n_ticks"     : len(grp),
    })

results_df = pd.DataFrame(results)

print("=" * 55)
print(f"  PBV Strategy — {STOCK} — {SIDE}")
print("=" * 55)
print(f"  Total minutes simulated : {len(results_df)}")
print(f"  Fallback executions     : {results_df['fallback'].sum()} "
      f"({100*results_df['fallback'].mean():.1f}%)")
print(f"  Mean improvement vs TWAP: {results_df['improvement'].mean():.5f}")
print(f"  Median improvement      : {results_df['improvement'].median():.5f}")
print(f"  % minutes beating TWAP  : {100*(results_df['improvement'] > 0).mean():.1f}%")
print(f"  Mean exec second        : {results_df['exec_second'].mean():.1f}s")
print()
results_df.describe()

In [ ]:
# ============================================================
# CELL 7: Results Visualisation
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 1. Improvement per minute (sorted)
axes[0].bar(
    range(len(results_df)),
    results_df["improvement"].sort_values().values,
    color=["steelblue" if v >= 0 else "tomato"
           for v in results_df["improvement"].sort_values().values],
    width=1.0, edgecolor="none"
)
axes[0].axhline(0, color="black", linewidth=0.8)
axes[0].set_title("Improvement over TWAP (per minute, sorted)")
axes[0].set_xlabel("Minute rank")
axes[0].set_ylabel("Price improvement ($)")

# 2. Histogram of improvement
axes[1].hist(results_df["improvement"], bins=40, color="steelblue", edgecolor="white")
axes[1].axvline(0, color="black", linewidth=1)
axes[1].axvline(results_df["improvement"].mean(), color="red",
                linewidth=1.5, linestyle="--", label=f"mean={results_df['improvement'].mean():.4f}")
axes[1].set_title("Distribution of improvement")
axes[1].set_xlabel("Price improvement ($)")
axes[1].legend()

# 3. Execution timing
axes[2].hist(results_df["exec_second"], bins=30, color="darkorange", edgecolor="white")
axes[2].axvline(FALLBACK_SECONDS, color="red", linewidth=1.5,
                linestyle="--", label=f"fallback threshold ({FALLBACK_SECONDS}s)")
axes[2].set_title("Execution timing within minute")
axes[2].set_xlabel("Seconds from minute start")
axes[2].legend()

plt.suptitle(f"PBV Strategy — {STOCK} — {SIDE}", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 8: Single-Minute Deep Dive — Signal Overlay
# ============================================================
# Visualise one minute in detail: bucket vectors + 2nd derivatives + where signal fired.

INSPECT_MINUTE_IDX = 5   # change to inspect a different minute
inspect_minute = train["Minute"].unique()[INSPECT_MINUTE_IDX]
sample = train[train["Minute"] == inspect_minute].reset_index(drop=True)
sample["signal"] = sample.apply(SIGNAL_FN, axis=1)

fired_idx = sample[sample["signal"]].index[0] if sample["signal"].any() else len(sample) - 1

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Mid price
axes[0].plot(sample["MidPrice"].values, color="black", linewidth=1.2, label="MidPrice")
axes[0].axvline(fired_idx, color="green", linestyle="--", linewidth=1.5, label="PBV exec")
axes[0].axvline(0, color="red", linestyle=":", linewidth=1.5, label="TWAP exec (tick 0)")
axes[0].set_title(f"MidPrice — {inspect_minute}")
axes[0].legend(fontsize=8)

# Cumulative bucket vectors
colors = ["black", "red", "salmon", "steelblue", "lightblue"]
for col, c in zip(BUCKET_COLS, colors):
    axes[1].plot(sample[f"{col}_cum"].values, label=col, color=c, alpha=0.8)
axes[1].axvline(fired_idx, color="green", linestyle="--", linewidth=1.5)
axes[1].axhline(0, color="gray", linewidth=0.5, linestyle="--")
axes[1].set_title("Cumulative bucket vectors (within-minute state)")
axes[1].legend(fontsize=7, ncol=5)

# 2nd derivatives
for col, c in zip(BUCKET_COLS, colors):
    axes[2].plot(sample[f"{col}_d2"].values, label=col, color=c, alpha=0.8)
axes[2].axvline(fired_idx, color="green", linestyle="--", linewidth=1.5)
axes[2].axhline(0, color="gray", linewidth=0.5, linestyle="--")
axes[2].set_title("2nd derivative (acceleration) of bucket vectors")
axes[2].set_xlabel("Tick within minute")
axes[2].legend(fontsize=7, ncol=5)

plt.suptitle(f"Deep dive — {STOCK} — {inspect_minute} — Signal fired at tick {fired_idx}", fontsize=12)
plt.tight_layout()
plt.show()

print(f"TWAP price : {sample.iloc[0][EXEC_PRICE_COL]:.4f}")
print(f"PBV price  : {sample.iloc[fired_idx][EXEC_PRICE_COL]:.4f}")
print(f"Improvement: {(sample.iloc[0][EXEC_PRICE_COL] - sample.iloc[fired_idx][EXEC_PRICE_COL]):.4f}")

## Next Steps

### Signal Refinement
- **Threshold tuning**: replace hard-coded `< 0` comparisons with adaptive thresholds (e.g. rolling percentile of d2 within the minute so far)
- **Far bucket signals**: add `v_ask_far_d2` and `v_bid_far_d2` — outer book acceleration often leads inner book moves
- **Combined score**: weighted sum of all 5 bucket signals into a single scalar, then threshold that
- **Minimum hold period**: require N consecutive signal ticks before executing (noise filter)

### Feature Extensions
- **Spread interaction**: down-weight signal when spread is wide (execution more costly)
- **Trade flow**: incorporate `Direction_1=Buy_-1=Sell` to separate passive vs aggressive activity per bucket
- **Cross-level momentum**: slope of `v_ask_near_cum` vs `v_ask_far_cum` — divergence often precedes price move

### Scaling to All Stocks
- Set `STOCK = "GOOG"` / `"INTC"` / `"MSFT"` in Cell 1 and re-run
- Wrap everything in a loop over stocks and collect per-stock improvement stats
- Stock-specific signal calibration may be needed (INTC has very different tick size / liquidity profile)

### Evaluation
- Run on **out-of-sample test data** once strategy is tuned on train
- Report: mean improvement per share, % minutes beating TWAP, per-stock breakdown